# Rolling-origin validation — is the result stable, or was September lucky?

`merchant_ranking_hybrid.ipynb` and `learned_weights.ipynb` both end on a single untouched month,
September 2022. That is honest, but it is one month: a single large merchant having an unusual
September moves the numbers more than it should.

This notebook repeats the same test across **four windows that walk forward through time**, each
with a **three-month** target rather than one. If the ranking's advantage survives all four, it is
not an artefact of September. If it does not, that is the more valuable finding, and better that we
report it than that it is asked about in the presentation.

**What the data allows.** Transactions run 28 Feb 2021 – 26 Oct 2022, but February holds a single
day and October stops on the 26th. Only **March 2021 – September 2022** are complete months, so
every window below sits inside that range and no partial month is ever used as a target.

| Fold | Features built from | Target (3 months) | Weights learned on |
| --- | --- | --- | --- |
| **A** | Mar – Sep 2021 | Oct – Dec 2021 | — (no earlier window) |
| **B** | Mar – Dec 2021 | Jan – Mar 2022 | fold A's pair |
| **C** | Mar 2021 – Mar 2022 | Apr – Jun 2022 | fold B's pair |
| **FINAL** | Mar 2021 – Jun 2022 | Jul – Sep 2022 | fold C's pair |

**The rule that keeps this clean:** a fold's model may only be fitted on a window whose target ends
*before* that fold's target begins. Fold A has no such window, so no learned model is scored there —
only the hand weights and the baseline, which need no fitting.

The features expand as time passes, which is how the firm would actually work: every month that
arrives is one more month of evidence.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "tables" / "processed"

FOLDS = [
    {"name": "A",     "feat": ("2021-03-01", "2021-09-30"), "months": 7,
     "label": ("2021-10-01", "2021-12-31")},
    {"name": "B",     "feat": ("2021-03-01", "2021-12-31"), "months": 10,
     "label": ("2022-01-01", "2022-03-31")},
    {"name": "C",     "feat": ("2021-03-01", "2022-03-31"), "months": 13,
     "label": ("2022-04-01", "2022-06-30")},
    {"name": "FINAL", "feat": ("2021-03-01", "2022-06-30"), "months": 16,
     "label": ("2022-07-01", "2022-09-30")},
]

spark = (
    SparkSession.builder.appName("MAST30034 Project 2 - Rolling validation")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

transactions = spark.read.parquet(str(PROCESSED / "transactions_clean"))
merchants = spark.read.parquet(str(PROCESSED / "merchants_clean"))
consumers = spark.read.parquet(str(PROCESSED / "consumers_sa2"))

print("folds:")
for f in FOLDS:
    print(f"  {f['name']:<6} features {f['feat'][0]} to {f['feat'][1]} "
          f"({f['months']} months)  ->  target {f['label'][0]} to {f['label'][1]}")

## 1. Build each fold once

Four feature tables and four target tables. The definitions are identical to
`merchant_ranking_hybrid.ipynb`; only the dates move. Building these is the slow part of the notebook —
everything after it is arithmetic on about 4,000 rows.

In [ ]:
def build_features(tx, months_in_window):
    core = (
        tx.groupBy("merchant_abn")
        .agg(F.sum("dollar_value").alias("gmv"),
             F.count("*").alias("transaction_count"),
             F.countDistinct("user_id").alias("unique_customers"),
             F.countDistinct(F.trunc("order_datetime", "month")).alias("active_months"))
        .withColumn("transactions_per_customer",
                    F.col("transaction_count") / F.col("unique_customers"))
        .withColumn("active_month_ratio", F.col("active_months") / F.lit(months_in_window))
    )

    per_cust = (tx.groupBy("merchant_abn", "user_id")
                  .agg(F.sum("dollar_value").alias("cust_gmv"),
                       F.count("*").alias("cust_orders")))

    conc = (per_cust.groupBy("merchant_abn")
            .agg((F.sum(F.when(F.col("cust_orders") > 1, 1).otherwise(0)) / F.count("*"))
                    .alias("repeat_rate"),
                 F.sum(F.pow("cust_gmv", 2)).alias("_sq"),
                 F.sum("cust_gmv").alias("_tot"))
            .withColumn("customer_hhi", F.col("_sq") / F.pow(F.col("_tot"), 2))
            .drop("_sq", "_tot"))

    monthly = (tx.withColumn("month", F.trunc("order_datetime", "month"))
                 .groupBy("merchant_abn", "month")
                 .agg(F.sum("dollar_value").alias("monthly_gmv")))

    stab = (monthly.groupBy("merchant_abn")
            .agg(F.avg("monthly_gmv").alias("mean_monthly_value"),
                 F.stddev_pop("monthly_gmv").alias("sd_monthly_value"))
            .withColumn("value_cv",
                        F.col("sd_monthly_value") / F.col("mean_monthly_value")))

    w_asc = Window.partitionBy("merchant_abn").orderBy(F.asc("month"))
    w_desc = Window.partitionBy("merchant_abn").orderBy(F.desc("month"))
    rm = (monthly.withColumn("i_early", F.row_number().over(w_asc))
                 .withColumn("i_late", F.row_number().over(w_desc)))

    growth = (rm.groupBy("merchant_abn")
        .agg(F.sum(F.when(F.col("i_early") <= 3, F.col("monthly_gmv"))).alias("early_3m_value"),
             F.sum(F.when(F.col("i_late") <= 3, F.col("monthly_gmv"))).alias("recent_3m_value"),
             F.count("*").alias("n_months"))
        .withColumn("log_value_growth",
                    F.when(F.col("n_months") >= 6,
                           F.log1p("recent_3m_value") - F.log1p("early_3m_value")))
        .drop("n_months"))

    cp = per_cust.join(
        consumers.select("user_id", "sa2_share",
                         "abs_median_household_income_weekly", "seifa_irsad_decile"),
        on="user_id", how="left")
    hi = F.col("abs_median_household_income_weekly").isNotNull()
    hr = F.col("seifa_irsad_decile").isNotNull()
    wt = F.coalesce(F.col("sa2_share"), F.lit(1.0))

    base = (cp.groupBy("merchant_abn").agg(
        (F.sum(F.when(hi, F.col("abs_median_household_income_weekly") * wt))
         / F.sum(F.when(hi, wt))).alias("customer_income"),
        (F.sum(F.when(hr, F.col("seifa_irsad_decile") * wt))
         / F.sum(F.when(hr, wt))).alias("customer_irsad")))

    return (core.join(conc, "merchant_abn", "left")
                .join(stab.select("merchant_abn", "value_cv"), "merchant_abn", "left")
                .join(growth, "merchant_abn", "left")
                .join(base, "merchant_abn", "left")
                .join(merchants.select("merchant_abn", "name", "take_rate"),
                      "merchant_abn", "left")
                .withColumn("estimated_bnpl_revenue",
                            F.col("gmv") * F.col("take_rate") / 100.0))


def realised(tx):
    return (tx.groupBy("merchant_abn")
              .agg(F.sum("dollar_value").alias("label_gmv"),
                   F.count("*").alias("label_transactions"))
              .join(merchants.select("merchant_abn", "take_rate"), "merchant_abn", "left")
              .withColumn("realised_revenue", F.col("label_gmv") * F.col("take_rate") / 100.0)
              .select("merchant_abn", "label_transactions", "realised_revenue"))


frames = {}
for fold in FOLDS:
    feats = build_features(
        transactions.filter(F.col("order_datetime").between(*fold["feat"])), fold["months"]
    ).filter(F.col("take_rate").isNotNull())
    lab = realised(transactions.filter(F.col("order_datetime").between(*fold["label"])))
    d = feats.join(lab, "merchant_abn", "left").toPandas()
    d[["label_transactions", "realised_revenue"]] = \
        d[["label_transactions", "realised_revenue"]].fillna(0.0)
    frames[fold["name"]] = d
    print(f"{fold['name']:<6} {len(d):>6,} merchants   "
          f"${d['realised_revenue'].sum()/1e6:>7,.1f}M realised in the target window   "
          f"{int((d['label_transactions'] == 0).sum()):>4} stopped trading")

## 2. The pickers

Three contenders, scored identically in every fold.

- **hand weights** — fixed numbers, nothing is fitted, so they can be scored in all four folds.
- **Learned non-negative weights** — refitted inside every fold on the *previous* fold's pair, so
  the model never sees the window it is judged on. Fold A has no previous pair and is left blank.
- **Hybrid weights** — the shape the learned model keeps choosing, with two deliberate departures
  from it. The machine sets the socio-economic features to zero in every fold; they are held at a
  floor of 5% because the brief requires external data to inform the ranking, and because a feature
  that cannot predict one quarter of revenue may still describe exposure that matters over the life
  of a merchant relationship. Growth keeps a token 2% for the same reason. Nothing is fitted here,
  so it is scored in all four folds, and §3 reports what the two departures cost.
- **Baseline: past revenue** — the benchmark that has beaten us everywhere so far.

Perfect foresight is the denominator for the capture rate, not a contender.

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

WEIGHTS_HAND = {
    "estimated_bnpl_revenue":    0.70,
    "log_value_growth":          0.08,
    "transactions_per_customer": 0.04,
    "repeat_rate":               0.03,
    "stability":                 0.03,
    "active_month_ratio":        0.02,
    "customer_spread":           0.05,
    "customer_income":           0.03,
    "customer_irsad":            0.02,
}
# the shape the learned model settles on across folds, with floors for the brief's requirements
WEIGHTS_HYBRID = {
    "estimated_bnpl_revenue":    0.70,   # machine: ~0.72
    "active_month_ratio":        0.18,   # machine: ~0.22 - the correction we missed
    "customer_spread":           0.05,   # machine: ~0.06
    "customer_income":           0.03,   # machine: 0 - held at a floor
    "customer_irsad":            0.02,   # machine: 0 - held at a floor
    "log_value_growth":          0.02,   # machine: ~0 - token weight
}
assert abs(sum(WEIGHTS_HYBRID.values()) - 1) < 1e-9, sum(WEIGHTS_HYBRID.values())

FEATURES = list(WEIGHTS_HAND)
SHRINK = {"estimated_bnpl_revenue", "log_value_growth"}
K = 300


def design(df):
    d = df.copy()
    d["stability"] = -d["value_cv"]
    d["customer_spread"] = 1 - d["customer_hhi"]
    shrinkage = d["transaction_count"] / (d["transaction_count"] + K)
    R = pd.DataFrame(
        {c: d[c].rank(pct=True, na_option="keep") * 100 for c in FEATURES}
    ).fillna(50.0)
    for c in SHRINK:
        R[c] = shrinkage * R[c] + (1 - shrinkage) * 50.0
    return R


def judge(frame, series):
    perfect = frame.nlargest(100, "realised_revenue")
    best = perfect["realised_revenue"].sum()
    sel = frame.assign(_s=series).nlargest(100, "_s")
    return {
        "spearman": spearmanr(series, frame["realised_revenue"])[0],
        "capture rate": sel["realised_revenue"].sum() / best,
        "hits": len(set(sel["merchant_abn"]) & set(perfect["merchant_abn"])),
        "stopped": int((sel["label_transactions"] == 0).sum()),
    }


rows, learned_by_fold = [], {}

for i, fold in enumerate(FOLDS):
    name = fold["name"]
    frame = frames[name]
    X = design(frame)

    rows.append({"fold": name, "picker": "hand weights",
                 **judge(frame, sum(X[c] * w for c, w in WEIGHTS_HAND.items()))})

    rows.append({"fold": name, "picker": "hybrid: evidence + ABS floor",
                 **judge(frame, sum(X[c] * w for c, w in WEIGHTS_HYBRID.items()))})

    if i > 0:  # fit on the previous fold's pair - it ends before this target begins
        prev = frames[FOLDS[i - 1]["name"]]
        model = LinearRegression(positive=True).fit(
            design(prev), np.log1p(prev["realised_revenue"]))
        coef = pd.Series(model.coef_, index=FEATURES)
        learned_by_fold[name] = coef / coef.sum() if coef.sum() > 0 else coef
        rows.append({"fold": name, "picker": "learned non-negative",
                     **judge(frame, pd.Series(model.predict(X), index=frame.index))})

    rows.append({"fold": name, "picker": "baseline: past revenue",
                 **judge(frame, frame["estimated_bnpl_revenue"].rank(pct=True) * 100)})

results = pd.DataFrame(rows)
results.round(3)

## 3. Does the ordering of the three hold up?

The table below is the point of the whole notebook: the same comparison, four times, on four
different stretches of time.

Read the **spread** as carefully as the average. A picker that wins by eight points in one fold and
loses by six in another is not better — it is unstable, and an average hides that.

In [ ]:
capture = results.pivot(index="picker", columns="fold", values="capture rate")
capture = capture[[f["name"] for f in FOLDS]]
capture["mean"] = capture.mean(axis=1)
capture["worst"] = capture.min(axis=1)

print("CAPTURE RATE per fold\n")
print((capture * 100).round(1).to_string())

spear = results.pivot(index="picker", columns="fold", values="spearman")[
    [f["name"] for f in FOLDS]]
spear["mean"] = spear.mean(axis=1)
print("\n\nSPEARMAN per fold\n")
print(spear.round(3).to_string())

hits = results.pivot(index="picker", columns="fold", values="hits")[
    [f["name"] for f in FOLDS]]
print("\n\nHITS in the realised top 100 (out of 100)\n")
print(hits.to_string())

### Did the learned weights themselves stay still?

Weights refitted in each fold are three independent answers to the same question. Weights that jump
around between folds are fitting noise, whatever their score; weights that keep roughly the same
shape are describing something real about merchants.

In [ ]:
if learned_by_fold:
    w_folds = pd.DataFrame(learned_by_fold)
    w_folds["mean"] = w_folds.mean(axis=1)
    w_folds["hand"] = pd.Series(WEIGHTS_HAND)
    print("Learned non-negative weights, refitted per fold (normalised to sum to 1)\n")
    print(w_folds.round(3).sort_values("mean", ascending=False).to_string())
else:
    print("no fold had an earlier window to learn from")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"hand weights": "#F58518",
          "hybrid: evidence + ABS floor": "#4C78A8",
          "learned non-negative": "#54A24B",
          "baseline: past revenue": "#8C8C8C"}

fold_names = [f["name"] for f in FOLDS]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

x = np.arange(len(fold_names))
width = 0.21
for j, picker in enumerate(COLORS):
    vals = [results.query("fold == @f and picker == @picker")["capture rate"].mean() * 100
            for f in fold_names]
    axes[0].bar(x + (j - 1.5) * width, vals, width=width, color=COLORS[picker], label=picker)
axes[0].set_xticks(x); axes[0].set_xticklabels(fold_names)
axes[0].set_ylabel("capture rate (%)"); axes[0].set_ylim(0, 108)
axes[0].set_title("Capture rate across four walk-forward windows")
axes[0].legend(frameon=False, fontsize=8, loc="lower left")

for picker, color in COLORS.items():
    sub = results[results["picker"] == picker].set_index("fold").reindex(fold_names)
    axes[1].plot(fold_names, sub["spearman"], marker="o", color=color, label=picker)
axes[1].set_ylabel("Spearman"); axes[1].set_title("Rank correlation across the same windows")
axes[1].legend(frameon=False, fontsize=8, loc="lower left")

plt.tight_layout(); plt.show()

## 4. How to report this

Write the sentence that the table actually supports, not the one we hoped for:

- **If the order of the three pickers is the same in all four folds**, say so plainly: *"the
  comparison was repeated across four walk-forward windows with three-month targets, and the
  ordering did not change."* That is a much stronger claim than any single holdout, and it is the
  version a marker cannot poke at.
- **If the order changes between folds**, the honest sentence is that the differences between these
  rankings are within the noise of this dataset — which, given that past revenue has been close to
  perfect everywhere, is a perfectly respectable finding.

**The decision this notebook is meant to settle**

Compare the hybrid against the learned weights across the four folds. If the gap is under about a
point, the floors on the ABS features and growth are cheap, the hybrid is the version to ship, and
the report can state its price exactly. If the gap is large, the floors are expensive and the group
has a real trade-off to argue rather than a free one.

**Limitations that stay true either way**

1. **Folds overlap.** The feature windows expand, so fold C contains fold B's data. The targets never
   overlap, which is what matters for leakage, but the folds are not four independent experiments.
2. **Fold A is the weakest.** Seven months of features is barely enough for `log_value_growth`
   (it needs six active months), and no model is scored there at all.
3. **One dataset, one firm, synthetic consumers.** Everything here describes this data, not BNPL
   in general.
4. **The fraud penalty is not applied in this notebook.** These are pre-risk rankings, exactly as
   in `merchant_ranking_hybrid.ipynb`.

In [ ]:
OUT = PROCESSED / "rolling_validation_results.csv"
results.to_csv(OUT, index=False)

OUT_W = PROCESSED / "rolling_learned_weights.csv"
if learned_by_fold:
    pd.DataFrame(learned_by_fold).to_csv(OUT_W)
    print(f"written: {OUT.name}, {OUT_W.name}")
else:
    print(f"written: {OUT.name}")

winner = (results.groupby("picker")["capture rate"].mean().sort_values(ascending=False))
print("\nAverage capture rate across folds:")
print((winner * 100).round(1).to_string())
if {"hybrid: evidence + ABS floor", "learned non-negative"} <= set(winner.index):
    shared = [f["name"] for f in FOLDS if f["name"] != "A"]   # learned has no fold A
    sub = results[results["fold"].isin(shared)]
    gap = (sub.query("picker == 'learned non-negative'")["capture rate"].mean()
           - sub.query("picker == 'hybrid: evidence + ABS floor'")["capture rate"].mean())
    print(f"\nPrice of the ABS and growth floors, folds {', '.join(shared)}: "
          f"{gap * 100:+.2f} capture-rate points")
